<a href="https://colab.research.google.com/github/maymay-wa/Bio-Project/blob/main/src/Maya/SimpleVersion%20Mayer%20Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
!git clone https://github.com/maymay-wa/Bio-Project.git
!cd /content/Bio-Project/src
!unzip -n /content/Bio-Project/Data/training_data.zip -d /content/Bio-Project/Data

fatal: destination path 'Bio-Project' already exists and is not an empty directory.
Archive:  /content/Bio-Project/Data/training_data.zip


In [21]:
import sys
sys.path.insert(0, "../Tools")

import pandas
import numpy as np
import tensorflow as tf
from pathlib import Path
from dna_affinity_data import DNAAffinityDataProcessor

In [26]:
# Initialize and process the data
processor = DNAAffinityDataProcessor(
    sequences_path="/content/Bio-Project/Data//training_seqs.txt",
    dbps_path="/content/Bio-Project/Data//training_DBPs.txt",
    affinity_path="/content/Bio-Project/Data//training_data.txt",
)
df_affinity = processor.process()

FileNotFoundError: [Errno 2] No such file or directory: '/content/Bio-Project/Data/training_data.txt'

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(df_affinity["Affinity"], bins=100, color="#4C78A8", edgecolor="white", alpha=0.9)
plt.title("Affinity Distribution")
plt.xlabel("Affinity")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Build model inputs.
dna = np.stack(df_affinity["DNA_onehot"].to_list(), axis=0).astype(np.float32)
proteins = df_affinity["Protein"].astype(str).to_numpy()
labels = df_affinity["Affinity"].to_numpy(dtype=np.float32)

# Train/validation split.
rng = np.random.default_rng(42)
indices = rng.permutation(len(labels))
split = int(0.8 * len(labels))
train_idx, val_idx = indices[:split], indices[split:]

dna_train, dna_val = dna[train_idx], dna[val_idx]
prot_train, prot_val = proteins[train_idx], proteins[val_idx]
y_train, y_val = labels[train_idx], labels[val_idx]

# Protein string lookup and embedding.
protein_lookup = tf.keras.layers.StringLookup(output_mode="int")
protein_lookup.adapt(prot_train)
protein_vocab_size = protein_lookup.vocabulary_size()

In [ ]:
dna_input = tf.keras.Input(shape=(36, 4), name="dna_onehot")
prot_input = tf.keras.Input(shape=(), dtype=tf.string, name="protein")

x_dna = tf.keras.layers.Conv1D(32, 5, padding="same", activation="relu")(dna_input)
x_dna = tf.keras.layers.BatchNormalization()(x_dna)
x_dna = tf.keras.layers.MaxPooling1D(2)(x_dna)
x_dna = tf.keras.layers.Conv1D(64, 3, padding="same", activation="relu")(x_dna)
x_dna = tf.keras.layers.GlobalMaxPooling1D()(x_dna)

x_prot = protein_lookup(prot_input)
x_prot = tf.keras.layers.Embedding(protein_vocab_size, 8)(x_prot)
x_prot = tf.keras.layers.Flatten()(x_prot)

x = tf.keras.layers.Concatenate()([x_dna, x_prot])
x = tf.keras.layers.Dense(64, activation="relu")(x)
x = tf.keras.layers.Dropout(0.2)(x)
output = tf.keras.layers.Dense(1, name="affinity")(x)

In [ ]:
def within_tolerance(y_true, y_pred, tol=0.5):
    return tf.reduce_mean(tf.cast(tf.abs(y_true - y_pred) <= tol, tf.float32))

model = tf.keras.Model(inputs=[dna_input, prot_input], outputs=output)
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae", tf.keras.metrics.RootMeanSquaredError(name="rmse"), within_tolerance],
)
train_ds = tf.data.Dataset.from_tensor_slices(
    ({"dna_onehot": dna_train, "protein": prot_train}, y_train)
).shuffle(2048, seed=42).batch(128).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices(
    ({"dna_onehot": dna_val, "protein": prot_val}, y_val)
).batch(128).prefetch(tf.data.AUTOTUNE)
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    )
 ]

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=1,
 )

In [ ]:
# hello maya